# Colab Simulation: Gamma Hurdle (Oganisian et al. 2019) on DGP C: Tweedie Semicontinuous
- **Model**: Gamma Hurdle (Oganisian et al. 2019) (`Gamma-Hurdle`) — benchmark, shipped in `zicbcf`
- **DGP**: DGP C: Tweedie Semicontinuous
- **Sample Size (N)**: 500
- **Zero-Inflation Intercept (c_shift)**: 0.0
- **MCMC**: NBURN = 1000, NSIM = 1000
- **Simulations**: 100 seeds
- **Output**: CSV with CATE and Hurdle metrics; `Status` flags any failed seed.


In [ ]:
# Install the zicbcf package from GitHub (ships BCF, ZIC-BCF, and the
# EDP+ZI / DPglm benchmark models with all their dependencies).
install.packages("remotes", repos="https://cloud.r-project.org/")
if (!require("devtools")) {
  install.packages("devtools", repos="https://cloud.r-project.org/")
}
devtools::install_github("hugogobato/zicbcf")
library(zicbcf)

In [ ]:
N_SIM   <- 100L
N       <- 500L
P       <- 5L
NBURN   <- 1000L
NSIM    <- 1000L
NTHIN   <- 1L

OUT_CSV <- "results_gamma_hurdle_dgp_c_N500.csv"
if (file.exists(OUT_CSV)) file.remove(OUT_CSV)

In [ ]:
generate_dgp_c <- function(n, p, seed, c_shift = 0.0) {
  set.seed(seed * 1000 + 42)
  X <- matrix(rnorm(n * p), n, p)
  colnames(X) <- paste0("X", 1:p)

  pi_x <- pnorm(-0.5 + 0.4 * X[, 1] + 0.3 * X[, 2]^2)
  Z    <- rbinom(n, 1, pi_x)

  # Log-mean parameter (halved treatment effect)
  log_mu0 <- 1.2 + c_shift + 0.8 * X[, 1] - 0.4 * X[, 3]
  log_mu1 <- 1.2 + c_shift + 0.8 * X[, 1] - 0.4 * X[, 3] + 0.3 + 0.15 * X[, 1]

  mu0_true  <- exp(log_mu0)
  mu1_true  <- exp(log_mu1)
  true_cate <- mu1_true - mu0_true

  mu_true  <- ifelse(Z == 1, mu1_true, mu0_true)
  phi_true <- 1.5

  lambda0_true <- 2 * sqrt(mu0_true) / phi_true
  lambda1_true <- 2 * sqrt(mu1_true) / phi_true

  lambda_true <- ifelse(Z == 1, lambda1_true, lambda0_true)
  N_latent    <- rpois(n, lambda_true)
  gamma_true  <- 0.5 * phi_true * sqrt(mu_true)

  Y <- rep(0, n)
  for (i in 1:n) {
    if (N_latent[i] > 0) {
      Y[i] <- rgamma(1, shape = N_latent[i], scale = gamma_true[i])
    }
  }

  p0_hurdle_true <- 1 - exp(-lambda0_true)
  p1_hurdle_true <- 1 - exp(-lambda1_true)
  true_hurdle_cate <- p1_hurdle_true - p0_hurdle_true

  list(y = Y, z = Z, x = X, pihat = pi_x, true_cate = true_cate, true_ate = mean(true_cate),
       true_hurdle_cate = true_hurdle_cate, true_hurdle_ate = mean(true_hurdle_cate))
}
calc_cate_metrics <- function(cate_draws, true_c, ate_draws) {
  cate_est <- colMeans(cate_draws)
  cate_ci  <- apply(cate_draws, 2, quantile, probs = c(0.025, 0.975))

  rmse <- sqrt(mean((cate_est - true_c)^2))
  bias <- mean(cate_est - true_c)
  coverage <- mean(true_c >= cate_ci[1, ] & true_c <= cate_ci[2, ])
  correlation <- cor(cate_est, true_c)
  if (is.na(correlation)) correlation <- 0.0
  ci_length <- mean(cate_ci[2, ] - cate_ci[1, ])
  est_ate_mean <- mean(ate_draws)

  list(rmse=rmse, bias=bias, coverage=coverage, correlation=correlation, ci_length=ci_length, est_ate_mean=est_ate_mean)
}

In [ ]:
cat("=== Starting Gamma Hurdle (Oganisian et al. 2019) on DGP C: Tweedie Semicontinuous ===\n")
for (s in 1:N_SIM) {
  cat(sprintf("[Seed %d/%d] Generating and fitting...\n", s, N_SIM))
  d <- generate_dgp_c(N, P, seed = s, c_shift = 0.0)

  fit <- tryCatch(
    gamma_hurdle(y = d$y, z = d$z, x = d$x,
                      nburn = NBURN, nsim = NSIM, nthin = NTHIN),
    error = function(e) { message(sprintf("  seed %d FAILED: %s", s, conditionMessage(e))); NULL }
  )

  if (is.null(fit)) {
    # Record the failed seed transparently.
    df_res <- data.frame(
      DGP = "DGP C: Tweedie Semicontinuous", Seed = s, Status = "FAILED",
      True_ATE = d$true_ate, True_Hurdle_ATE = d$true_hurdle_ate,
      GammaHurdle_CATE_RMSE = NA,
      GammaHurdle_CATE_Abs_Bias = NA,
      GammaHurdle_CATE_Coverage = NA,
      GammaHurdle_CATE_Correlation = NA,
      GammaHurdle_CATE_CI_Length = NA,
      GammaHurdle_Est_ATE = NA,
      GammaHurdle_Hurdle_RMSE = NA,
      GammaHurdle_Hurdle_Abs_Bias = NA,
      GammaHurdle_Hurdle_Coverage = NA,
      GammaHurdle_Hurdle_Correlation = NA,
      GammaHurdle_Hurdle_CI_Length = NA,
      GammaHurdle_Est_Hurdle_ATE = NA,
      stringsAsFactors = FALSE)
  } else {
    m  <- calc_cate_metrics(fit$cate, d$true_cate, fit$ate)
    hurdle_cate_draws <- fit$p1 - fit$p0
    mh <- calc_cate_metrics(hurdle_cate_draws, d$true_hurdle_cate, rowMeans(hurdle_cate_draws))
    df_res <- data.frame(
      DGP = "DGP C: Tweedie Semicontinuous", Seed = s, Status = "OK",
      True_ATE = d$true_ate, True_Hurdle_ATE = d$true_hurdle_ate,
      GammaHurdle_CATE_RMSE = m$rmse,
      GammaHurdle_CATE_Abs_Bias = abs(m$bias),
      GammaHurdle_CATE_Coverage = m$coverage,
      GammaHurdle_CATE_Correlation = m$correlation,
      GammaHurdle_CATE_CI_Length = m$ci_length,
      GammaHurdle_Est_ATE = m$est_ate_mean,
      GammaHurdle_Hurdle_RMSE = mh$rmse,
      GammaHurdle_Hurdle_Abs_Bias = abs(mh$bias),
      GammaHurdle_Hurdle_Coverage = mh$coverage,
      GammaHurdle_Hurdle_Correlation = mh$correlation,
      GammaHurdle_Hurdle_CI_Length = mh$ci_length,
      GammaHurdle_Est_Hurdle_ATE = mh$est_ate_mean,
      stringsAsFactors = FALSE)
  }

  write.table(df_res, OUT_CSV, sep=",", row.names=FALSE,
              col.names=!file.exists(OUT_CSV), append=TRUE)
  gc(verbose=FALSE)
}
cat("\n=== Finished Gamma Hurdle (Oganisian et al. 2019) Run ===\n")